## 1. Environment Setup

In [1]:
!pip install unsloth transformers datasets accelerate peft bitsandbytes code-bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 24.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:

!pip install codebleu==0.7.0 tree-sitter==0.22.3 tree-sitter-java==0.21.0 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.2/546.2 kB 11.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


In [3]:
import gc
import json
import logging
import os
import random
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
from datasets import Dataset, load_dataset

print("Core imports ready | torch", torch.__version__,
      "| CUDA", torch.cuda.is_available())

Core imports ready | torch 2.10.0+cu128 | CUDA True


## 2. Configuration

In [4]:
@dataclass(frozen=True)
class Config:
    """Single source of truth for every tunable value in the notebook."""

    SEED: int = 3407

    MODEL_NAME: str = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"
    MAX_LENGTH: int = 2048

    DATASET_NAME: str = "code_search_net"
    DATASET_CONFIG: str = "java"
    STAGE1_TRAIN_SAMPLES: int = 5000
    STAGE1_OUTPUT_DIR: str = "~/nl2java_checkpoints_V5"

    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.0
    LORA_BIAS: str = "none"
    LORA_TARGET_MODULES: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
    )

    EPOCHS: int = 3
    BATCH_SIZE: int = 4
    GRAD_ACCUM: int = 4
    LEARNING_RATE: float = 2e-5
    SAVE_STRATEGY: str = "epoch"
    LOGGING_STEPS: int = 20

    STAGE2_MAX_SEQ_LEN: int = 1024
    STAGE2_MAX_SAMPLES: int = 6000
    STAGE2_EPOCHS: int = 2
    STAGE2_BATCH_SIZE: int = 2
    STAGE2_GRAD_ACCUM: int = 4
    STAGE2_LEARNING_RATE: float = 2e-4
    STAGE2_LORA_ALPHA: int = 16

    EVAL_SAMPLES: int = 100
    STAGE2_EVAL_SAMPLES: int = 50

    SAVE_DIR: str = "./saved_models"
    HF_REPO: str = "shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5"
    HF_REPO_STAGE2: str = "shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5"


CFG = Config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Configuration loaded. DEVICE={DEVICE} | run timestamp={TIMESTAMP}")

Configuration loaded. DEVICE=cuda | run timestamp=20260719_043327


In [5]:
def configure_logging(name: str = "QwenCodeGen", level: int = logging.INFO) -> logging.Logger:
    """Configure a structured root logger and return a named child logger.

    Format: ``timestamp | LEVEL | name | message``. Safe to call repeatedly
    (``force=True`` resets existing handlers so re-running the cell behaves).
    """
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
        force=True,
    )
    return logging.getLogger(name)


logger = configure_logging()


def set_global_seed(seed: int) -> None:
    """Seed Python, NumPy, PyTorch (and HF) RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        from transformers import set_seed as hf_set_seed
        hf_set_seed(seed)
    except Exception:
        pass
    logger.info("Global seed set to %d", seed)


def log_gpu_info(tag: str = "") -> None:
    """Log GPU name + current memory usage, or warn if running on CPU."""
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        logger.info(
            "GPU [%s] %s | %.2f GB allocated, %.2f GB reserved, %.1f GB total",
            tag, name, alloc, reserved, total,
        )
    else:
        logger.warning(
            "CUDA not available — running on CPU (training will be very slow).")


def free_memory(*names: str, scope: dict | None = None) -> None:
    """Delete named globals, run GC and empty the CUDA cache."""
    scope = scope if scope is not None else globals()
    for n in names:
        if n in scope:
            del scope[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log_gpu_info("after free_memory")


set_global_seed(CFG.SEED)
log_gpu_info("startup")

04:33:44 | WARNING | torchao | Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
04:33:46 | INFO | QwenCodeGen | Global seed set to 3407
04:33:46 | INFO | QwenCodeGen | GPU [startup] Tesla T4 | 0.00 GB allocated, 0.00 GB reserved, 15.6 GB total


## 3. Stage 1 – Dataset Preparation

In [7]:
STAGE1_DOC_COL = "func_documentation_string"
STAGE1_CODE_COL = "func_code_string"


def load_code_dataset(split: str,
                      dataset_name: str = CFG.DATASET_NAME,
                      config: str = CFG.DATASET_CONFIG):
    """Load a split of the code dataset with logging and error handling."""
    try:
        logger.info("Loading %s/%s split='%s' ...",
                    dataset_name, config, split)
        ds = load_dataset(dataset_name, config, split=split)
        logger.info("Loaded %d examples for split '%s'.", len(ds), split)
        return ds
    except Exception:
        logger.exception("Failed to load dataset %s/%s (split=%s).",
                         dataset_name, config, split)
        raise


def validate_dataset(ds, required_columns: Sequence[str], n_check: int = 100) -> None:
    """Validate that required columns exist and have non-empty values."""
    if ds is None or len(ds) == 0:
        raise ValueError("Dataset is empty or None.")
    missing = [c for c in required_columns if c not in ds.column_names]
    if missing:
        raise ValueError(f"Dataset missing required columns: {missing} "
                         f"(available: {ds.column_names})")
    sample = ds.select(range(min(n_check, len(ds))))
    for col in required_columns:
        n_null = sum(1 for v in sample[col] if v is None or not str(v).strip())
        if n_null:
            logger.warning("Column '%s' has %d empty values in first %d rows.",
                           col, n_null, len(sample))
    logger.info("Dataset validation passed for columns: %s",
                list(required_columns))


dataset = load_code_dataset("train")
validate_dataset(dataset, [STAGE1_DOC_COL, STAGE1_CODE_COL])

04:33:55 | INFO | QwenCodeGen | Loading code_search_net/java split='train' ...
04:33:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
04:33:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
04:33:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"
04:33:56 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

04:33:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 307 Temporary Redirect"
04:33:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 404 Not Found"
04:33:56 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/code_search_net/code_search_net.py "HTTP/1.1 404 Not Found"
04:33:56 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/code_search_net/revision/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555 "HTTP/1.1 307 Temporary Redirect"
04:33:56 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/code-search-net/code_search_net/revision/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555 "HTTP/1.1 200 OK"
04:33:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/dat

java/train-00000-of-00001.parquet:   0%|          | 0.00/390M [00:00<?, ?B/s]

04:34:02 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/java/test-00000-of-00001.parquet "HTTP/1.1 307 Temporary Redirect"
04:34:02 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/java/test-00000-of-00001.parquet "HTTP/1.1 302 Found"


...

java/validation-00000-of-00001.parquet:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

04:34:08 | INFO | QwenCodeGen | Loaded 454451 examples for split 'train'.
04:34:08 | INFO | QwenCodeGen | Dataset validation passed for columns: ['func_documentation_string', 'func_code_string']


In [8]:
logger.info("Available columns: %s", dataset.column_names)
_example = dataset[2]
logger.info("Doc (truncated to instruction): %s",
            _example[STAGE1_DOC_COL].split("@param")[0].strip()[:300])
logger.info("Code (first 300 chars):\n%s", _example[STAGE1_CODE_COL][:300])

04:34:12 | INFO | QwenCodeGen | Available columns: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url']
04:34:12 | INFO | QwenCodeGen | Doc (truncated to instruction): Set Contrast adjusting factor, [-127, 127].
04:34:12 | INFO | QwenCodeGen | Code (first 300 chars):
public void setFactor(int factor) {
        this.factor = factor = Math.max(-127, Math.min(127, factor));

        if (factor > 1) {
            baseFilter.setInRed(new IntRange(factor, 255 - factor));
            baseFilter.setInGreen(new IntRange(factor, 255 - factor));
            baseFilte


## 4. Stage 1 – Training

In [9]:
from unsloth import FastLanguageModel


def load_base_model(model_name: str = CFG.MODEL_NAME,
                    max_seq_length: int = CFG.MAX_LENGTH,
                    load_in_4bit: bool = True):
    """Load a 4-bit base model + tokenizer via Unsloth, with logging/validation."""
    try:
        logger.info("Loading base model '%s' (max_seq_length=%d, 4bit=%s) ...",
                    model_name, max_seq_length, load_in_4bit)
        mdl, tok = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=load_in_4bit,
        )
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        logger.info("Base model loaded. EOS token id=%s", tok.eos_token_id)
        log_gpu_info("after base model load")
        return mdl, tok
    except Exception:
        logger.exception("Failed to load base model '%s'.", model_name)
        raise

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [10]:
model, tokenizer = load_base_model()

04:34:39 | INFO | QwenCodeGen | Loading base model 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit' (max_seq_length=2048, 4bit=True) ...
04:34:39 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:34:39 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
04:34:39 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
04:34:39 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


04:34:39 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
04:34:40 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Temporary Redirect"
04:34:40 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 200 OK"
04:34:40 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 200 OK"
04:34:40 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 307 Temporary Redirect"
04:34:40 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/model.safet

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

04:34:51 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
04:34:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/generation_config.json "HTTP/1.1 200 OK"
04:34:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:34:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
04:34:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04:34:52 | INFO | httpx | HTTP Req

In [11]:
def configure_lora(base_model,
                   r: int = CFG.LORA_R,
                   lora_alpha: int = CFG.LORA_ALPHA,
                   lora_dropout: float = CFG.LORA_DROPOUT,
                   bias: str = CFG.LORA_BIAS,
                   target_modules: Sequence[str] = CFG.LORA_TARGET_MODULES):
    """Attach a LoRA adapter to ``base_model`` and log trainable parameters."""
    try:
        logger.info("Configuring LoRA (r=%d, alpha=%d, dropout=%s) ...",
                    r, lora_alpha, lora_dropout)
        peft_model = FastLanguageModel.get_peft_model(
            model=base_model,
            r=r,
            lora_alpha=lora_alpha,
            target_modules=list(target_modules),
            lora_dropout=lora_dropout,
            bias=bias,
        )
        peft_model.print_trainable_parameters()
        logger.info("LoRA adapter attached.")
        return peft_model
    except Exception:
        logger.exception("Failed to configure LoRA adapter.")
        raise


model = configure_lora(model)

04:35:02 | INFO | QwenCodeGen | Configuring LoRA (r=16, alpha=32, dropout=0.0) ...
Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
04:35:07 | INFO | QwenCodeGen | LoRA adapter attached.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [12]:
STAGE1_RESPONSE_MARKER = "### Response:\n\n"

STAGE1_LANG_HINT = " Write the solution in Java."


def add_java_hint(instruction: str) -> str:
    """Append the Java hint unless the instruction already mentions Java."""
    instruction = instruction.strip()
    if "java" in instruction.lower():
        return instruction
    return f"{instruction}{STAGE1_LANG_HINT}"


def build_stage1_prompt(instruction: str) -> str:
    """Prompt up to (and including) the response marker — used at inference time."""
    return f"### Instruction:\n\n{add_java_hint(instruction)}\n\n{STAGE1_RESPONSE_MARKER}"


def clean_instruction(doc: str) -> str:
    """Extract the human-readable instruction from a javadoc string."""
    return doc.split("@param")[0].strip()


def build_stage1_training_text(doc: str, code: str, eos_token: str) -> str:
    """Full prompt + target Java code + EOS — used for SFT training.
    The EOS token is appended so the model learns where the Java answer ends;
    without it the model never emits EOS at inference and keeps generating
    unrelated code until it hits max_new_tokens.
    """
    return build_stage1_prompt(clean_instruction(doc)) + code + eos_token

In [13]:
def build_prompt_dataset(ds, doc_col: str = STAGE1_DOC_COL, code_col: str = STAGE1_CODE_COL,
                         eos_token: Optional[str] = None):
    """Batched preprocessing: build the `prompt` column and drop everything else."""
    # EOS is appended so the model learns where the Java answer ends (mirrors Stage 2).
    if eos_token is None:
        eos_token = tokenizer.eos_token or "<|endoftext|>"

    def _batch(batch):
        return {"prompt": [build_stage1_training_text(d, c, eos_token)
                           for d, c in zip(batch[doc_col], batch[code_col])]}

    try:
        logger.info(
            "Building prompt dataset (batched) over %d examples ...", len(ds))
        out = ds.map(_batch, batched=True,
                     remove_columns=ds.column_names,
                     desc="Building NL->Java prompts")
        logger.info("Prompt dataset ready: %d examples, columns=%s",
                    len(out), out.column_names)
        return out
    except Exception:
        logger.exception("Failed to build prompt dataset.")
        raise


prompt_dataset = build_prompt_dataset(dataset)
logger.info("Sample prompt:\n%s", prompt_dataset[2]["prompt"][:500])

04:35:33 | INFO | QwenCodeGen | Building prompt dataset (batched) over 454451 examples ...


Building NL->Java prompts:   0%|          | 0/454451 [00:00<?, ? examples/s]

04:35:36 | INFO | QwenCodeGen | Prompt dataset ready: 454451 examples, columns=['prompt']
04:35:36 | INFO | QwenCodeGen | Sample prompt:
### Instruction:

Set Contrast adjusting factor, [-127, 127]. Write the solution in Java.

### Response:

public void setFactor(int factor) {
        this.factor = factor = Math.max(-127, Math.min(127, factor));

        if (factor > 1) {
            baseFilter.setInRed(new IntRange(factor, 255 - factor));
            baseFilter.setInGreen(new IntRange(factor, 255 - factor));
            baseFilter.setInBlue(new IntRange(factor, 255 - factor));
            baseFilter.setInGray(new IntRang


In [14]:
import trl
from trl import SFTConfig, SFTTrainer

logger.info("Using trl version %s", trl.__version__)

04:35:39 | INFO | QwenCodeGen | Using trl version 0.24.0


In [ ]:
def create_stage1_config() -> SFTConfig:
    """Build the Stage 1 SFT configuration from CFG (FP16 only when a GPU is present)."""
    return SFTConfig(
        output_dir=CFG.STAGE1_OUTPUT_DIR,
        num_train_epochs=CFG.EPOCHS,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        gradient_accumulation_steps=CFG.GRAD_ACCUM,
        learning_rate=CFG.LEARNING_RATE,
        save_strategy=CFG.SAVE_STRATEGY,
        logging_steps=CFG.LOGGING_STEPS,
        dataset_text_field="prompt",
        fp16=torch.cuda.is_available(),
        seed=CFG.SEED,
        report_to="none",
    )


stage1_config = create_stage1_config()
logger.info("Stage 1 SFTConfig ready: epochs=%d, batch=%d, grad_accum=%d, lr=%s, fp16=%s",
            stage1_config.num_train_epochs, stage1_config.per_device_train_batch_size,
            stage1_config.gradient_accumulation_steps, stage1_config.learning_rate,
            stage1_config.fp16)

04:35:42 | INFO | QwenCodeGen | Stage 1 SFTConfig ready: epochs=3, batch=4, grad_accum=4, lr=2e-05, fp16=True


In [ ]:
def create_stage1_trainer(peft_model, tok, cfg, train_ds, n_samples: int = CFG.STAGE1_TRAIN_SAMPLES):
    """Create the Stage 1 SFTTrainer over the first ``n_samples`` examples."""
    n = min(n_samples, len(train_ds))
    if n <= 0:
        raise ValueError("No training samples available for Stage 1.")
    logger.info("Creating Stage 1 trainer over %d training samples.", n)
    common = dict(model=peft_model, args=cfg,
                  train_dataset=train_ds.select(range(n)))
    try:
        return SFTTrainer(tokenizer=tok, **common)
    except TypeError:
        return SFTTrainer(processing_class=tok, **common)


stage1_trainer = create_stage1_trainer(
    model, tokenizer, stage1_config, prompt_dataset)

04:35:45 | INFO | QwenCodeGen | Creating Stage 1 trainer over 5000 training samples.
04:35:45 | INFO | unsloth.trainer | Unsloth: Padding-free batching auto-enabled for SFTTrainer instance.


Unsloth: Tokenizing ["prompt"] (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [17]:
def train_stage1(trainer):
    """Run Stage 1 training with logging + graceful failure."""
    try:
        logger.info("Starting Stage 1 training (NL -> Java) ...")
        log_gpu_info("before Stage 1 train")
        stats = trainer.train()
        logger.info("Stage 1 training complete. %s",
                    stats.metrics if hasattr(stats, "metrics") else stats)
        log_gpu_info("after Stage 1 train")
        return stats
    except Exception:
        logger.exception("Stage 1 training failed.")
        raise


stage1_stats = train_stage1(stage1_trainer)

04:36:10 | INFO | QwenCodeGen | Starting Stage 1 training (NL -> Java) ...
04:36:10 | INFO | QwenCodeGen | GPU [before Stage 1 train] Tesla T4 | 1.27 GB allocated, 1.29 GB reserved, 15.6 GB total
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 3 | Total steps = 939
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,1.864378
40,1.780514
60,1.635915
80,1.496164
100,1.362013
120,1.281567
140,1.243459
160,1.176603
180,1.190308
200,1.104635


04:49:19 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:49:19 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
04:49:19 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:49:19 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
Unsloth: Restored added_tokens_decoder metadata in /root/nl2java_checkpoints_V5/checkpoint-313/tokenizer_config.json.
04:49:19 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4b

## 5. Stage 1 – Inference

In [ ]:
@torch.inference_mode()
def _run_generation(gen_model, gen_tokenizer, prompt_text: str, max_new_tokens: int) -> str:
    """Greedy generation primitive shared by both stages."""
    FastLanguageModel.for_inference(gen_model)
    inputs = gen_tokenizer(
        prompt_text, return_tensors="pt").to(gen_model.device)
    output = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=gen_tokenizer.eos_token_id,
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    return gen_tokenizer.decode(output[0], skip_special_tokens=True)


def generate_java_from_nl(prompt: str, max_new_tokens: int = 300) -> str:
    """Stage 1: natural language -> Java. Validates input and strips the prompt echo."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string.")
    full_prompt = build_stage1_prompt(prompt)
    try:
        decoded = _run_generation(
            model, tokenizer, full_prompt, max_new_tokens)
    except Exception:
        logger.exception(
            "NL -> Java generation failed for prompt: %r", prompt[:80])
        raise
    return decoded.split(STAGE1_RESPONSE_MARKER)[-1].strip()

In [19]:
_demo_prompts = [
    "write a function to check if a number is even",
    "write a function to implement binary search in Java",
    "write a function to find all permutations of a string",
]
for _p in _demo_prompts:
    logger.info("NL prompt: %s", _p)
    print(f"### {_p}\n{generate_java_from_nl(_p)}\n{'-' * 80}")

05:16:44 | INFO | QwenCodeGen | NL prompt: write a function to check if a number is even
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.mask

### write a function to check if a number is even
public static boolean isEven(int n) {
        return (n & 1) == 0;
    }
--------------------------------------------------------------------------------


05:16:49 | INFO | QwenCodeGen | NL prompt: write a function to find all permutations of a string
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### write a function to implement binary search in Java
public static <T extends Comparable<T>> int binarySearch(T[] array, T key) {
        return binarySearch(array, 0, array.length - 1, key);
    }
--------------------------------------------------------------------------------
### write a function to find all permutations of a string
public static List<String> getPermutations(String str) {
        if (str == null || str.length() == 0) {
            return new ArrayList<>();
        }
        char[] chars = str.toCharArray();
        List<String> result = new ArrayList<>();
        permute(chars, 0, chars.length - 1, result);
        return result;
    }
--------------------------------------------------------------------------------


## 6. Stage 1 – Evaluation

In [ ]:
def generate_predictions(gen_model, gen_tokenizer, subset,
                         max_new_tokens: int = 300):
    """Generate Java predictions + collect references for a test subset."""
    preds: List[str] = []
    refs: List[str] = []
    for i, d in enumerate(subset):
        prompt = build_stage1_prompt(clean_instruction(d[STAGE1_DOC_COL]))
        try:
            decoded = _run_generation(
                gen_model, gen_tokenizer, prompt, max_new_tokens)
            pred = decoded.split(STAGE1_RESPONSE_MARKER)[-1].strip()
        except Exception:
            logger.exception(
                "Generation failed at index %d; using empty prediction.", i)
            pred = ""
        preds.append(pred)
        refs.append(d[STAGE1_CODE_COL])
        if (i + 1) % 25 == 0:
            logger.info("  generated %d/%d", i + 1, len(subset))
    return preds, refs


CODEBLEU_WEIGHTS_NO_DATAFLOW = (1 / 3, 1 / 3, 1 / 3, 0.0)


def compute_code_metrics(references, predictions, lang: str = "java", label: str = ""):
    """CodeBLEU + CodeBERTScore for code predictions, robust to metric failures."""
    from codebleu import calc_codebleu
    from code_bert_score import score
    report: Dict[str, object] = {}
    try:
        report["CodeBLEU"] = calc_codebleu(references, predictions, lang=lang)
        # Dataflow-excluded variant: fairer when candidates are short/degenerate.
        report["CodeBLEU_no_dataflow"] = calc_codebleu(
            references, predictions, lang=lang,
            weights=CODEBLEU_WEIGHTS_NO_DATAFLOW)
    except Exception as exc:
        logger.exception("CodeBLEU computation failed.")
        report["CodeBLEU"] = {"error": str(exc)}
        report["CodeBLEU_no_dataflow"] = {"error": str(exc)}
    try:
        # code_bert_score.score expects (candidates, references) == (predictions, references).
        _, _, f1, _ = score(predictions, references, lang=lang)
        report["CodeBERTScore_F1"] = round(float(f1.mean()), 4)
    except Exception:
        logger.exception("CodeBERTScore computation failed.")
        report["CodeBERTScore_F1"] = None
    logger.info("[%s] Stage 1 metrics: %s", label, report)
    return report


base_model, base_tokenizer = load_base_model()

05:17:23 | INFO | QwenCodeGen | Loading base model 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit' (max_seq_length=2048, 4bit=True) ...
05:17:23 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:17:23 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
05:17:23 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


05:17:23 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
05:17:23 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 307 Temporary Redirect"
05:17:23 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
05:17:24 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
05:17:24 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Temporary Redirect"
05:17:24 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.git

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

05:17:29 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
05:17:29 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/generation_config.json "HTTP/1.1 200 OK"
05:17:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:17:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
05:17:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05:17:30 | INFO | httpx | HTTP Req

In [21]:
for _p in ["write a function to check if a number is even",
           "write a function to implement binary search in Java"]:
    _prompt = build_stage1_prompt(_p)
    _decoded = _run_generation(
        base_model, base_tokenizer, _prompt, max_new_tokens=300)
    print(
        f"### [BASE] {_p}\n{_decoded.split(STAGE1_RESPONSE_MARKER)[-1].strip()}\n{'-' * 80}")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### [BASE] write a function to check if a number is even
```java
public class EvenChecker {
    /**
     * Checks if the given number is even.
     *
     * @param number The number to check.
     * @return true if the number is even, false otherwise.
     */
    public static boolean isEven(int number) {
        return number % 2 == 0;
    }

    public static void main(String[] args) {
        // Test cases to verify the correctness of the isEven function
        System.out.println(isEven(4)); // Expected: true
        System.out.println(isEven(5)); // Expected: false
        System.out.println(isEven(10)); // Expected: true
        System.out.println(isEven(-2)); // Expected: true
        System.out.println(isEven(0)); // Expected: true
    }
}
```

This code snippet defines a class `EvenChecker` with a method `isEven` that takes an integer as input and returns `true` if the number is even, and `false` otherwise. The `main` method includes test cases to demonstrate the functionality

In [22]:
test_data = load_code_dataset("test")
validate_dataset(test_data, [STAGE1_DOC_COL, STAGE1_CODE_COL])
test_subset = test_data.select(range(min(CFG.EVAL_SAMPLES, len(test_data))))
logger.info("Evaluating on %d test examples.", len(test_subset))

05:18:48 | INFO | QwenCodeGen | Loading code_search_net/java split='test' ...
05:18:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
05:18:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
05:18:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"
05:18:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 307 Temporary Redirect"
05:18:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 404 Not Found"
05:18:48 | IN

In [23]:
logger.info("Generating fine-tuned predictions ...")
predictions, references = generate_predictions(model, tokenizer, test_subset)

05:18:58 | INFO | QwenCodeGen | Generating fine-tuned predictions ...
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (h

In [24]:
finetuned_metrics = compute_code_metrics(
    references, predictions, lang="java", label="fine-tuned")
print("Fine-tuned CodeBLEU:", finetuned_metrics["CodeBLEU"])
print("Fine-tuned CodeBERTScore F1:", finetuned_metrics["CodeBERTScore_F1"])

05:25:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:25:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/config.json "HTTP/1.1 200 OK"
05:25:53 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

05:25:54 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05:25:54 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"
05:25:54 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

05:25:54 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
05:25:54 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
05:25:54 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
05:25:54 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/vocab.json "HTTP/1.1 200 OK"
05:25:54 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/vocab.json "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

05:26:01 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/main "HTTP/1.1 200 OK"
05:26:01 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/discussions?p=0 "HTTP/1.1 200 OK"
05:26:01 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
05:26:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
05:26:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
05:26:01 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/xet-read-token/3bb7912eb9e672e30f3b2de2fadc31e8d2cbc66b "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

05:26:07 | INFO | QwenCodeGen | [fine-tuned] Stage 1 metrics: {'CodeBLEU': {'codebleu': 0.28294296839568667, 'ngram_match_score': 0.15331898297885893, 'weighted_ngram_match_score': 0.16693357003394765, 'syntax_match_score': 0.2653180885370859, 'dataflow_match_score': 0.5462012320328542}, 'CodeBLEU_no_dataflow': {'codebleu': 0.19519021384996416, 'ngram_match_score': 0.15331898297885893, 'weighted_ngram_match_score': 0.16693357003394765, 'syntax_match_score': 0.2653180885370859, 'dataflow_match_score': 0.5462012320328542}, 'CodeBERTScore_F1': 0.7857}


Fine-tuned CodeBLEU: {'codebleu': 0.28294296839568667, 'ngram_match_score': 0.15331898297885893, 'weighted_ngram_match_score': 0.16693357003394765, 'syntax_match_score': 0.2653180885370859, 'dataflow_match_score': 0.5462012320328542}
Fine-tuned CodeBERTScore F1: 0.7857


In [25]:
logger.info("Generating base-model predictions ...")
base_predictions, base_references = generate_predictions(
    base_model, base_tokenizer, test_subset)

05:26:12 | INFO | QwenCodeGen | Generating base-model predictions ...
05:31:48 | INFO | QwenCodeGen |   generated 25/100
05:37:22 | INFO | QwenCodeGen |   generated 50/100
05:42:52 | INFO | QwenCodeGen |   generated 75/100
05:48:24 | INFO | QwenCodeGen |   generated 100/100


In [26]:
base_metrics = compute_code_metrics(
    base_references, base_predictions, lang="java", label="base")
print("Base CodeBLEU:", base_metrics["CodeBLEU"])
print("Base CodeBERTScore F1:", base_metrics["CodeBERTScore_F1"])

stage1_evaluation = {"fine_tuned": finetuned_metrics, "base": base_metrics}
print("\n===== Stage 1 (NL -> Java) comparison =====")
print(f"Fine-tuned CodeBERTScore F1: {finetuned_metrics['CodeBERTScore_F1']}")
print(f"Base       CodeBERTScore F1: {base_metrics['CodeBERTScore_F1']}")

free_memory("base_model", "base_tokenizer")

05:48:33 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:48:33 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/config.json "HTTP/1.1 200 OK"
05:48:33 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05:48:33 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"
05:48:33 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
05:48:33 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main?recursive=true&e

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

05:48:34 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/main "HTTP/1.1 200 OK"
05:48:34 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/discussions?p=0 "HTTP/1.1 200 OK"
05:48:34 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
05:48:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
05:48:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
05:48:38 | INFO | QwenCodeGen | [base] Stage 1 metrics: {'CodeBLEU': {'codebleu': 0.260115720951056, 'ngram_match_score': 0.0022754356737825165, 'weighted_ngram_match_score': 0.016741251292936174, 'syntax_match_score': 0.19495749047200234, 'dataflow_match_score': 0.8264887063655031}

Base CodeBLEU: {'codebleu': 0.260115720951056, 'ngram_match_score': 0.0022754356737825165, 'weighted_ngram_match_score': 0.016741251292936174, 'syntax_match_score': 0.19495749047200234, 'dataflow_match_score': 0.8264887063655031}
Base CodeBERTScore F1: 0.6489

===== Stage 1 (NL -> Java) comparison =====
Fine-tuned CodeBERTScore F1: 0.7857
Base       CodeBERTScore F1: 0.6489


In [ ]:
def _extract_codebleu(metrics):
    """Pull the raw CodeBLEU dict from a metrics report, if present."""
    cb = metrics.get("CodeBLEU", {})
    return cb if isinstance(cb, dict) and "codebleu" in cb else {}


ft_cb, bs_cb = _extract_codebleu(
    finetuned_metrics), _extract_codebleu(base_metrics)
ft_nd = finetuned_metrics.get("CodeBLEU_no_dataflow", {})
bs_nd = base_metrics.get("CodeBLEU_no_dataflow", {})

print("===== Stage 1 (NL -> Java): fine-tuned vs base =====\n")
print(f"{'metric':32s}{'fine-tuned':>14s}{'base':>14s}")
print("-" * 60)
for k in ("ngram_match_score", "weighted_ngram_match_score",
          "syntax_match_score", "dataflow_match_score", "codebleu"):
    if k in ft_cb and k in bs_cb:
        print(f"{k:32s}{ft_cb[k]:>14.4f}{bs_cb[k]:>14.4f}")

print("-" * 60)
if "codebleu" in ft_nd and "codebleu" in bs_nd:
    print(f"{'codebleu (dataflow excluded)':32s}"
          f"{ft_nd['codebleu']:>14.4f}{bs_nd['codebleu']:>14.4f}")
print(f"{'CodeBERTScore F1':32s}"
      f"{(finetuned_metrics['CodeBERTScore_F1'] or float('nan')):>14.4f}"
      f"{(base_metrics['CodeBERTScore_F1'] or float('nan')):>14.4f}")

# Base model "wins" raw CodeBLEU only via an inflated dataflow score on
# near-empty output, so prefer the dataflow-excluded score / CodeBERTScore.
print("\nFine-tuned wins on n-gram, weighted-ngram, syntax match and CodeBERTScore.")

===== Stage 1 (NL -> Java): fine-tuned vs base =====

metric                              fine-tuned          base
------------------------------------------------------------
ngram_match_score                       0.1533        0.0023
weighted_ngram_match_score              0.1669        0.0167
syntax_match_score                      0.2653        0.1950
dataflow_match_score                    0.5462        0.8265
codebleu                                0.2829        0.2601
------------------------------------------------------------
codebleu (dataflow excluded)            0.1952        0.0713
CodeBERTScore F1                        0.7857        0.6489

Interpretation: the fine-tuned model wins on n-gram, weighted-ngram and syntax match (and CodeBERTScore). The base model only 'wins' raw CodeBLEU via an inflated dataflow score on near-empty output. Prefer the dataflow-excluded CodeBLEU and CodeBERTScore for a fair comparison.


## 7. Stage 2 – Dataset Preparation

In [30]:
!pip install -q sacrebleu nltk evaluate tree-sitter-c-sharp==0.21.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.6/387.6 kB 8.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
CODEXGLUE_DATASET_KWARGS = dict(
    path="google/code_x_glue_cc_code_to_code_trans")
CODEXGLUE_JAVA_COL = "java"
CODEXGLUE_CS_COL = "cs"


def _normalize_split(ds, java_col: Optional[str], csharp_col: Optional[str]):
    """Return a list of {'java', 'cs'} dicts, or None if columns are not usable."""
    cols = set(ds.column_names)
    if java_col is None or java_col not in cols:
        java_col = next((c for c in cols if c.lower() in (
            "java", "java_code", "src", "source")), None)
    if csharp_col is None or csharp_col not in cols:
        csharp_col = next((c for c in cols if c.lower() in (
            "cs", "csharp", "c#", "cs_code", "tgt", "target")), None)
    if not java_col or not csharp_col:
        return None
    pairs = [{"java": j, "cs": c}
             for j, c in zip(ds[java_col], ds[csharp_col])
             if isinstance(j, str) and isinstance(c, str) and j.strip() and c.strip()]
    return pairs or None


def load_java_csharp_dataset():
    """Load the Java<->C# dataset (CodeXGLUE-Java-CS)."""
    label = "CodeXGLUE-Java-CS"
    try:
        logger.info("Loading Java/C# dataset: %s (%s) ...",
                    label, CODEXGLUE_DATASET_KWARGS["path"])
        raw = load_dataset(**CODEXGLUE_DATASET_KWARGS)
        split_names = list(raw.keys()) if hasattr(raw, "keys") else ["train"]
        collected = {}
        for split in split_names:
            pairs = _normalize_split(
                raw[split], CODEXGLUE_JAVA_COL, CODEXGLUE_CS_COL)
            if pairs:
                collected[split] = pairs
        if not collected:
            raise RuntimeError(f"{label} has no usable Java/C# columns.")
        total = sum(len(v) for v in collected.values())
        logger.info("Loaded %s: %d Java/C# pairs across splits %s",
                    label, total, list(collected))
        return label, collected
    except Exception:
        logger.exception("Failed to load %s dataset.", label)
        raise


DATASET_NAME, raw_pairs = load_java_csharp_dataset()
logger.info("Selected Stage 2 dataset: %s", DATASET_NAME)

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

05:50:32 | INFO | QwenCodeGen | Trying Java/C# dataset: CodeXGLUE-Java-CS (google/code_x_glue_cc_code_to_code_trans) ...
05:50:32 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/google/code_x_glue_cc_code_to_code_trans/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
05:50:32 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/google/code_x_glue_cc_code_to_code_trans/d5478a4e472b5aae1c160f1b540ae2eebb79b640/README.md "HTTP/1.1 200 OK"
05:50:32 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/google/code_x_glue_cc_code_to_code_trans/d5478a4e472b5aae1c160f1b540ae2eebb79b640/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

05:50:32 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/google/code_x_glue_cc_code_to_code_trans/resolve/d5478a4e472b5aae1c160f1b540ae2eebb79b640/code_x_glue_cc_code_to_code_trans.py "HTTP/1.1 404 Not Found"
05:50:32 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/google/code_x_glue_cc_code_to_code_trans/google/code_x_glue_cc_code_to_code_trans.py "HTTP/1.1 404 Not Found"
05:50:32 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/google/code_x_glue_cc_code_to_code_trans/revision/d5478a4e472b5aae1c160f1b540ae2eebb79b640 "HTTP/1.1 200 OK"
05:50:32 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/google/code_x_glue_cc_code_to_code_trans/resolve/d5478a4e472b5aae1c160f1b540ae2eebb79b640/.huggingface.yaml "HTTP/1.1 404 Not Found"
05:50:32 | INFO | httpx | HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=google/code_x_glue_cc_code_to_code_trans "HTTP/1.1 200

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/170k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10300 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

05:50:35 | INFO | QwenCodeGen | Loaded CodeXGLUE-Java-CS: 11800 Java/C# pairs across splits ['train', 'validation', 'test']
05:50:35 | INFO | QwenCodeGen | Selected Stage 2 dataset: CodeXGLUE-Java-CS


In [33]:
STAGE2_INSTRUCTION = "Translate the following Java code into equivalent C#."
STAGE2_RESPONSE_MARKER = "### Response\n"
STAGE2_LANG_HINT = " Write the solution in C#."


def add_csharp_hint(instruction: str) -> str:
    """Append the C# hint unless the instruction already mentions C#."""
    instruction = instruction.strip()
    lowered = instruction.lower()
    if "c#" in lowered or "csharp" in lowered:
        return instruction
    return f"{instruction}{STAGE2_LANG_HINT}"


def build_stage2_inference_prompt(java_code: str) -> str:
    """Prompt up to (and including) the response marker — used at inference time."""
    return (
        "### Instruction\n"
        f"{add_csharp_hint(STAGE2_INSTRUCTION)}\n\n"
        "### Java\n"
        f"{java_code.strip()}\n\n"
        "### Response\n"
    )


def build_stage2_training_text(java_code: str, csharp_code: str, eos_token: str) -> str:
    """Full prompt + target + EOS — used for SFT training."""
    return build_stage2_inference_prompt(java_code) + csharp_code.strip() + eos_token

In [ ]:
random.seed(CFG.SEED)

all_pairs = []
for split_pairs in raw_pairs.values():
    all_pairs.extend(split_pairs)

logger.info("Raw Java/C# pairs: %d", len(all_pairs))


def is_valid_pair(java_code: str, csharp_code: str) -> bool:
    """Cheap quality filter: non-empty, not trivially short, and 'looks like code'."""
    j, c = java_code.strip(), csharp_code.strip()
    if not j or not c:
        return False
    if len(j) < 10 or len(c) < 10:
        return False
    if not any(tok in j for tok in (";", "{", "}")):
        return False
    if not any(tok in c for tok in (";", "{", "}")):
        return False
    return True


seen = set()
clean_pairs = []
for p in all_pairs:
    if not is_valid_pair(p["java"], p["cs"]):
        continue
    key = (p["java"].strip(), p["cs"].strip())
    if key in seen:
        continue
    seen.add(key)
    clean_pairs.append({"java": p["java"].strip(), "cs": p["cs"].strip()})

random.shuffle(clean_pairs)
logger.info("Clean, de-duplicated pairs: %d", len(clean_pairs))

clean_pairs = clean_pairs[:CFG.STAGE2_MAX_SAMPLES]

if len(clean_pairs) < 20:
    raise ValueError(
        f"Too few valid Java/C# pairs ({len(clean_pairs)}) to train Stage 2. "
        "Check the dataset loader / cleaning filters.")

val_size = min(max(50, int(0.05 * len(clean_pairs))),
               max(1, len(clean_pairs) // 5))
val_pairs = clean_pairs[:val_size]
train_pairs = clean_pairs[val_size:]
if not train_pairs:
    raise ValueError("Training split is empty after the validation cut.")
logger.info("Stage 2 split -> Train: %d | Validation: %d",
            len(train_pairs), len(val_pairs))

05:50:55 | INFO | QwenCodeGen | Raw Java/C# pairs: 11800
05:50:55 | INFO | QwenCodeGen | Clean, de-duplicated pairs: 11800
05:50:55 | INFO | QwenCodeGen | Stage 2 split -> Train: 5700 | Validation: 300


## 8. Stage 2 – Training

In [35]:
STAGE2_MAX_SEQ_LEN = CFG.STAGE2_MAX_SEQ_LEN

try:
    model_s2, tokenizer_s2 = FastLanguageModel.from_pretrained(
        model_name=CFG.MODEL_NAME,
        max_seq_length=STAGE2_MAX_SEQ_LEN,
        load_in_4bit=True,
    )
    if tokenizer_s2.pad_token is None:
        tokenizer_s2.pad_token = tokenizer_s2.eos_token

    model_s2 = FastLanguageModel.get_peft_model(
        model_s2,
        r=CFG.LORA_R,
        lora_alpha=CFG.STAGE2_LORA_ALPHA,
        lora_dropout=CFG.LORA_DROPOUT,
        bias=CFG.LORA_BIAS,
        target_modules=list(CFG.LORA_TARGET_MODULES),
        use_gradient_checkpointing="unsloth",
        max_seq_length=STAGE2_MAX_SEQ_LEN,
        random_state=CFG.SEED,
    )
    model_s2.print_trainable_parameters()
    logger.info("Stage 2 model + LoRA ready (separate adapter).")
    log_gpu_info("after Stage 2 model load")
except Exception:
    logger.exception("Failed to load Stage 2 model / attach LoRA adapter.")
    raise

05:51:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:51:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
05:51:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


05:51:04 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
05:51:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 307 Temporary Redirect"
05:51:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 200 OK"
05:51:04 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 200 OK"
05:51:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/model.safetensors "HTTP/1.1 302 Found"
05:51:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 307 T

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

05:51:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
05:51:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/generation_config.json "HTTP/1.1 200 OK"
05:51:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:51:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
05:51:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05:51:10 | INFO | httpx | HTTP Req

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [36]:
_eos = tokenizer_s2.eos_token or "<|endoftext|>"


def _to_text_dataset(pairs):
    texts = [build_stage2_training_text(
        p["java"], p["cs"], _eos) for p in pairs]
    return Dataset.from_dict({"text": texts})


train_dataset_s2 = _to_text_dataset(train_pairs)
val_dataset_s2 = _to_text_dataset(val_pairs)

logger.info("Stage 2 text datasets built: train=%d, val=%d",
            len(train_dataset_s2), len(val_dataset_s2))
logger.info("Example Stage 2 training sample:\n%s",
            train_dataset_s2[0]["text"][:800])

05:51:26 | INFO | QwenCodeGen | Stage 2 text datasets built: train=5700, val=300
05:51:26 | INFO | QwenCodeGen | Example Stage 2 training sample:
### Instruction
Translate the following Java code into equivalent C#.

### Java
public OldFormulaRecord(RecordInputStream ris) {super(ris, ris.getSid() == biff2_sid);if (isBiff2()) {field_4_value = ris.readDouble();} else {long valueLongBits  = ris.readLong();specialCachedValue = FormulaSpecialCachedValue.create(valueLongBits);if (specialCachedValue == null) {field_4_value = Double.longBitsToDouble(valueLongBits);}}if (isBiff2()) {field_5_options = (short)ris.readUByte();} else {field_5_options = ris.readShort();}int expression_len = ris.readShort();int nBytesAvailable = ris.available();field_6_parsed_expr = Formula.read(expression_len, ris, nBytesAvailable);}

### Response
public OldFormulaRecord(RecordInputStream ris) :base(ris, ris.Sid == biff2_sid){;if (IsBiff2){field_4_value = ris.Rea


In [ ]:
def pick_optimizer() -> str:
    """paged_adamw_8bit when bitsandbytes is available, else adamw_torch."""
    try:
        import bitsandbytes  # noqa: F401
        return "paged_adamw_8bit"
    except Exception:
        return "adamw_torch"


def _accepted_config_keys():
    keys = set(getattr(SFTConfig, "__dataclass_fields__", {}).keys())
    try:
        from transformers import TrainingArguments
        keys |= set(getattr(TrainingArguments,
                    "__dataclass_fields__", {}).keys())
    except Exception:
        pass
    return keys


def build_sft_config(**kwargs) -> "SFTConfig":
    """Construct an SFTConfig, tolerating version-specific keyword differences."""
    keys = _accepted_config_keys()
    if "eval_strategy" in kwargs and "eval_strategy" not in keys and "evaluation_strategy" in keys:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    if keys:
        dropped = [k for k in kwargs if k not in keys]
        for k in dropped:
            kwargs.pop(k)
        if dropped:
            logger.warning(
                "Dropped unsupported SFTConfig args for this version: %s", dropped)
    return SFTConfig(**kwargs)


def build_sft_trainer(cfg):
    """Create the SFTTrainer, tolerating the tokenizer -> processing_class rename."""
    common = dict(
        model=model_s2,
        args=cfg,
        train_dataset=train_dataset_s2,
        eval_dataset=val_dataset_s2,
    )
    try:
        return SFTTrainer(tokenizer=tokenizer_s2, **common)
    except TypeError:
        return SFTTrainer(processing_class=tokenizer_s2, **common)


use_fp16 = torch.cuda.is_available()

stage2_config = build_sft_config(
    output_dir="./java2csharp_checkpoints",
    num_train_epochs=CFG.STAGE2_EPOCHS,
    per_device_train_batch_size=CFG.STAGE2_BATCH_SIZE,
    gradient_accumulation_steps=CFG.STAGE2_GRAD_ACCUM,
    learning_rate=CFG.STAGE2_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=CFG.LOGGING_STEPS,
    save_strategy="epoch",
    eval_strategy="epoch",
    optim=pick_optimizer(),
    fp16=use_fp16,
    bf16=False,
    gradient_checkpointing=True,
    max_seq_length=STAGE2_MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
    seed=CFG.SEED,
)

stage2_trainer = build_sft_trainer(stage2_config)
logger.info("Stage 2 trainer ready. Optimizer=%s | fp16=%s",
            stage2_config.optim, stage2_config.fp16)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/5700 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/300 [00:00<?, ? examples/s]

05:52:09 | INFO | QwenCodeGen | Stage 2 trainer ready. Optimizer=OptimizerNames.PAGED_ADAMW_8BIT | fp16=True


In [40]:
try:
    logger.info("Starting Stage 2 training (Java -> C#) ...")
    log_gpu_info("before Stage 2 train")
    stage2_stats = stage2_trainer.train()
    logger.info("Stage 2 training complete. %s",
                stage2_stats.metrics if hasattr(stage2_stats, "metrics") else stage2_stats)
    log_gpu_info("after Stage 2 train")
except Exception:
    logger.exception("Stage 2 training failed.")
    raise

06:06:42 | INFO | QwenCodeGen | Starting Stage 2 training (Java -> C#) ...
06:06:42 | INFO | QwenCodeGen | GPU [before Stage 2 train] Tesla T4 | 2.76 GB allocated, 5.28 GB reserved, 15.6 GB total


{'loss': '0.4209', 'grad_norm': '0.5633', 'learning_rate': '8.837e-05', 'epoch': '0.02807'}
{'loss': '0.3642', 'grad_norm': '0.4894', 'learning_rate': '0.0001814', 'epoch': '0.05614'}
{'loss': '0.3197', 'grad_norm': '0.5221', 'learning_rate': '0.0001999', 'epoch': '0.08421'}
{'loss': '0.2967', 'grad_norm': '0.5735', 'learning_rate': '0.0001997', 'epoch': '0.1123'}
{'loss': '0.2785', 'grad_norm': '1.093', 'learning_rate': '0.0001992', 'epoch': '0.1404'}
{'loss': '0.2791', 'grad_norm': '0.5481', 'learning_rate': '0.0001985', 'epoch': '0.1684'}
{'loss': '0.3272', 'grad_norm': '0.5174', 'learning_rate': '0.0001976', 'epoch': '0.1965'}
{'loss': '0.412', 'grad_norm': '0.5768', 'learning_rate': '0.0001965', 'epoch': '0.2246'}
{'loss': '0.4136', 'grad_norm': '0.4907', 'learning_rate': '0.0001953', 'epoch': '0.2526'}
{'loss': '0.3471', 'grad_norm': '0.5871', 'learning_rate': '0.0001938', 'epoch': '0.2807'}
{'loss': '0.3756', 'grad_norm': '0.5277', 'learning_rate': '0.0001921', 'epoch': '0.3088'

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

{'eval_loss': '0.4184', 'eval_runtime': '16.86', 'eval_samples_per_second': '17.8', 'eval_steps_per_second': '4.45', 'epoch': '1'}


06:31:58 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
06:31:58 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:31:58 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"


...

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

{'eval_loss': '0.4127', 'eval_runtime': '15.98', 'eval_samples_per_second': '18.77', 'eval_steps_per_second': '4.693', 'epoch': '2'}


06:56:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:56:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"


{'train_runtime': '3004', 'train_samples_per_second': '3.795', 'train_steps_per_second': '0.475', 'train_loss': '0.3592', 'epoch': '2'}


06:56:49 | INFO | QwenCodeGen | Stage 2 training complete. {'train_runtime': 3004.1531, 'train_samples_per_second': 3.795, 'train_steps_per_second': 0.475, 'train_loss': 0.3591610873731958, 'epoch': 2.0}
06:56:49 | INFO | QwenCodeGen | GPU [after Stage 2 train] Tesla T4 | 2.69 GB allocated, 2.79 GB reserved, 15.6 GB total


## 9. Stage 2 – Inference

In [ ]:
import re

STAGE2_INFERENCE_GUARDRAILS = (
    " This is a literal, line-by-line translation task, not a code review: keep "
    "the exact same method name(s), return type(s), parameter names, parameter "
    "types, parameter order, and access modifiers as the Java input. Preserve the "
    "exact algorithm, all loops, conditions, recursion, numeric types, and "
    "exception handling exactly as written in the Java — do not fix bugs, "
    "optimize, simplify, or rewrite the logic, and do not introduce new helper "
    "methods. Only add 'virtual', 'override', 'async', 'sealed', or 'new' "
    "modifiers if the Java input's own modifiers or class hierarchy already "
    "require them; otherwise omit them."
)


def build_stage2_strict_inference_prompt(java_code: str) -> str:
    """Inference-only prompt: splices the guardrails into the shared training
    prompt right before the "### Java" section, without touching training."""
    base_prompt = build_stage2_inference_prompt(java_code)
    marker = "\n\n### Java\n"
    idx = base_prompt.index(marker)
    return base_prompt[:idx] + STAGE2_INFERENCE_GUARDRAILS + base_prompt[idx:]


def strip_unwanted_csharp_modifiers(csharp_code: str) -> str:
    """Drop C# modifiers the Java->C# translation tends to hallucinate, only
    when doing so is guaranteed safe: 'virtual'/'sealed' always; 'override'
    unless the snippet declares a base class/interface; 'async' unless it
    contains a matching 'await'. 'new' is left alone (ambiguous with `new T()`).
    """
    code = re.sub(r'\bvirtual\s+', '', csharp_code)
    code = re.sub(r'\bsealed\s+', '', code)

    has_inheritance = re.search(
        r'\bclass\s+\w+(<[^>]*>)?\s*:\s*\w', code) is not None
    if not has_inheritance:
        code = re.sub(r'\boverride\s+', '', code)

    if 'await' not in code:
        code = re.sub(r'\basync\s+', '', code)

    return code


def _generate_csharp(gen_model, gen_tokenizer, java_code: str,
                     max_new_tokens: int = 400) -> str:
    """Model-agnostic Java -> C# generation: strict prompt + post-processing.
    Shared by the fine-tuned and base-model evaluation paths."""
    if not isinstance(java_code, str) or not java_code.strip():
        raise ValueError("java_code must be a non-empty string.")
    full_prompt = build_stage2_strict_inference_prompt(java_code)
    decoded = _run_generation(
        gen_model, gen_tokenizer, full_prompt, max_new_tokens)
    generated = decoded.split(STAGE2_RESPONSE_MARKER)[-1].strip()
    return strip_unwanted_csharp_modifiers(generated)


def generate_csharp_from_java(java_code: str, max_new_tokens: int = 400) -> str:
    """Stage 2: Java -> C# (fine-tuned model_s2)."""
    try:
        return _generate_csharp(model_s2, tokenizer_s2, java_code, max_new_tokens)
    except Exception:
        logger.exception("Java -> C# generation failed.")
        raise


def generate_csharp_from_nl(prompt: str,
                            java_max_new_tokens: int = 300,
                            csharp_max_new_tokens: int = 400) -> Dict[str, str]:
    """Full chain: NL -> Java -> C#. Returns both intermediate Java and final C#."""
    java_code = generate_java_from_nl(
        prompt, max_new_tokens=java_max_new_tokens)
    csharp_code = generate_csharp_from_java(
        java_code, max_new_tokens=csharp_max_new_tokens)
    return {"natural_language": prompt, "java": java_code, "csharp": csharp_code}

In [53]:
print("=== Mode 1: NL -> Java ===")
print(generate_java_from_nl("write a function to check if a number is prime"))

print("\n=== Stage 2: Java -> C# ===")
sample_java = """public int add(int a, int b) {
    return a + b;
}"""
print(generate_csharp_from_java(sample_java))

print("\n=== Mode 2: NL -> Java -> C# (chained) ===")
result = generate_csharp_from_nl(
    "write a function to compute the factorial of a number")
print("\n--- Java ---\n", result["java"])
print("\n--- C# ---\n", result["csharp"])

=== Mode 1: NL -> Java ===
public static boolean isPrime(int n) {
        if (n < 2)
            return false;
        for (int i = 2; i <= Math.sqrt(n); i++)
            if (n % i == 0)
                return false;
        return true;
    }

=== Stage 2: Java -> C# ===
public int Add(int a, int b){return a + b;}

=== Mode 2: NL -> Java -> C# (chained) ===

--- Java ---
 public static long factorial(int n) {
        if (n < 0)
            throw new IllegalArgumentException("Factorial is not defined for negative numbers");
        else if (n == 0 || n == 1)
            return 1;
        else
            return n * factorial(n - 1);
    }

--- C# ---
 public static long Factorial(int n){if (n < 0){throw new ArgumentException("Factorial is not defined for negative numbers");}else{if (n == 0 || n == 1){return 1;}else{return n * Factorial(n - 1);}}}


## 10. Stage 2 – Evaluation

In [ ]:
import math
import re
import shutil
import subprocess
import tempfile
from collections import Counter

import sacrebleu
from codebleu import calc_codebleu


def _normalize_code(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())


def get_csharp_parser():
    """Best-effort tree-sitter C# parser; returns None if unavailable."""
    try:
        from tree_sitter import Language, Parser
        import tree_sitter_c_sharp as tscs
        lang = Language(tscs.language())
        try:
            return Parser(lang)
        except TypeError:
            p = Parser()
            p.set_language(lang)
            return p
    except Exception as exc:
        print("C# tree-sitter parser unavailable:", exc)
        return None


CS_PARSER = get_csharp_parser()


def _collect_node_types(node, acc):
    acc.append(node.type)
    for child in node.children:
        _collect_node_types(child, acc)


def _ast_node_types(code: str):
    if CS_PARSER is None:
        return None
    try:
        tree = CS_PARSER.parse(bytes(code, "utf8"))
        acc = []
        _collect_node_types(tree.root_node, acc)
        return acc
    except Exception:
        return None


def _parse_is_valid(code: str):
    if CS_PARSER is None:
        return None
    try:
        tree = CS_PARSER.parse(bytes(code, "utf8"))
        return not tree.root_node.has_error
    except Exception:
        return False


def _bag_cosine(a, b):
    ca, cb = Counter(a), Counter(b)
    keys = set(ca) | set(cb)
    dot = sum(ca[k] * cb[k] for k in keys)
    na = math.sqrt(sum(v * v for v in ca.values()))
    nb = math.sqrt(sum(v * v for v in cb.values()))
    return dot / (na * nb) if na and nb else 0.0


def _find_csharp_compiler():
    for tool in ("dotnet", "csc", "mcs"):
        if shutil.which(tool):
            return tool
    return None


CSHARP_COMPILER = _find_csharp_compiler()


def _wrap_csharp(code: str) -> str:
    """Wrap a bare snippet/method into a compilable class skeleton."""
    if re.search(r"\b(class|struct|interface|enum|namespace)\b", code):
        return code
    return ("using System;\nusing System.Collections.Generic;\n"
            "public class Wrapper {\n" + code + "\n}\n")


def _compiles_with_mono(code: str) -> bool:
    with tempfile.TemporaryDirectory() as tmp:
        src = f"{tmp}/Program.cs"
        with open(src, "w", encoding="utf-8") as fh:
            fh.write(_wrap_csharp(code))
        try:
            out = subprocess.run([CSHARP_COMPILER, "-target:library", src],
                                 capture_output=True, timeout=30)
            return out.returncode == 0
        except Exception:
            return False


def compilation_rate(preds):
    """Returns (rate, method_label)."""
    if CSHARP_COMPILER in ("csc", "mcs"):
        oks = [_compiles_with_mono(p) for p in preds]
        return sum(oks) / len(preds), f"real-compiler({CSHARP_COMPILER})"
    valids = [v for v in (_parse_is_valid(p) for p in preds) if v is not None]
    if not valids:
        return None, "unavailable"
    return sum(valids) / len(valids), "syntax-proxy(tree-sitter)"


def evaluate_csharp(predictions, references):
    report = {}

    report["BLEU"] = round(sacrebleu.corpus_bleu(
        predictions, [references]).score, 4)

    try:
        cb = calc_codebleu(references, predictions, lang="c_sharp")
        report["CodeBLEU"] = {k: round(v, 4) for k, v in cb.items()}
    except Exception as exc:
        report["CodeBLEU"] = {"error": str(exc)}

    report["ExactMatch"] = round(
        sum(_normalize_code(p) == _normalize_code(r)
            for p, r in zip(predictions, references)) / len(predictions), 4)

    syn = [v for v in (_parse_is_valid(p)
                       for p in predictions) if v is not None]
    report["SyntaxAccuracy"] = round(sum(syn) / len(syn), 4) if syn else None

    sims = []
    for p, r in zip(predictions, references):
        a, b = _ast_node_types(p), _ast_node_types(r)
        if a is not None and b is not None:
            sims.append(_bag_cosine(a, b))
    report["AST_Similarity"] = round(
        sum(sims) / len(sims), 4) if sims else None

    rate, method = compilation_rate(predictions)
    report["CompilationRate"] = {"value": (round(rate, 4) if rate is not None else None),
                                 "method": method}
    return report

In [ ]:
def _run_stage2_eval_loop(gen_fn, pairs, label: str = ""):
    """Run `gen_fn` (model-agnostic) over `pairs`, collecting predictions + references."""
    preds: List[str] = []
    refs: List[str] = []
    for i, p in enumerate(pairs):
        try:
            pred = gen_fn(p["java"], max_new_tokens=400)
        except Exception:
            logger.exception(
                "[%s] Stage 2 eval generation failed at index %d.", label, i)
            pred = ""
        preds.append(pred)
        refs.append(p["cs"])
        if (i + 1) % 10 == 0:
            logger.info("[%s] evaluated %d/%d", label, i + 1, len(pairs))
    return preds, refs


EVAL_N = min(CFG.STAGE2_EVAL_SAMPLES, len(val_pairs))
eval_pairs = val_pairs[:EVAL_N]

cs_predictions, cs_references = _run_stage2_eval_loop(
    generate_csharp_from_java, eval_pairs, label="fine-tuned")

stage2_report = evaluate_csharp(cs_predictions, cs_references)

logger.info("Stage 2 fine-tuned evaluation finished on %d samples (dataset=%s).",
            EVAL_N, DATASET_NAME)
print("\n===== Stage 2 (Java -> C#) Fine-tuned Evaluation Report =====")
print(f"Dataset: {DATASET_NAME} | Eval samples: {EVAL_N}")
print(json.dumps(stage2_report, indent=2))

06:58:39 | INFO | QwenCodeGen | [fine-tuned] evaluated 10/50
06:59:17 | INFO | QwenCodeGen | [fine-tuned] evaluated 20/50
06:59:52 | INFO | QwenCodeGen | [fine-tuned] evaluated 30/50
07:00:21 | INFO | QwenCodeGen | [fine-tuned] evaluated 40/50
07:00:59 | INFO | QwenCodeGen | [fine-tuned] evaluated 50/50
07:00:59 | INFO | QwenCodeGen | Stage 2 fine-tuned evaluation finished on 50 samples (dataset=CodeXGLUE-Java-CS).



===== Stage 2 (Java -> C#) Fine-tuned Evaluation Report =====
Dataset: CodeXGLUE-Java-CS | Eval samples: 50
{
  "BLEU": 83.2815,
  "CodeBLEU": {
    "codebleu": 0.7376,
    "ngram_match_score": 0.6837,
    "weighted_ngram_match_score": 0.68,
    "syntax_match_score": 0.7685,
    "dataflow_match_score": 0.8184
  },
  "ExactMatch": 0.40,
  "SyntaxAccuracy": 0.92,
  "AST_Similarity": 0.9817,
  "CompilationRate": {
    "value": 0.92,
    "method": "syntax-proxy(tree-sitter)"
  }
}


In [48]:
stage2_base_model, stage2_base_tokenizer = load_base_model(
    max_seq_length=STAGE2_MAX_SEQ_LEN)

07:01:15 | INFO | QwenCodeGen | Loading base model 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit' (max_seq_length=1024, 4bit=True) ...
07:01:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:01:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
07:01:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


07:01:16 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
07:01:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 307 Temporary Redirect"
07:01:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
07:01:16 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
07:01:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/model.safetensors "HTTP/1.1 302 Found"
07:01:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Te

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

07:01:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
07:01:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/generation_config.json "HTTP/1.1 200 OK"
07:01:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:01:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
07:01:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
07:01:22 | INFO | httpx | HTTP Req

In [49]:
logger.info("Generating Stage 2 base-model predictions ...")
stage2_base_predictions, stage2_base_references = _run_stage2_eval_loop(
    lambda java_code, max_new_tokens=400: _generate_csharp(
        stage2_base_model, stage2_base_tokenizer, java_code, max_new_tokens),
    eval_pairs, label="base")

stage2_base_report = evaluate_csharp(
    stage2_base_predictions, stage2_base_references)

logger.info("Stage 2 base-model evaluation finished on %d samples (dataset=%s).",
            EVAL_N, DATASET_NAME)
print("\n===== Stage 2 (Java -> C#) Base Model Evaluation Report =====")
print(f"Dataset: {DATASET_NAME} | Eval samples: {EVAL_N}")
print(json.dumps(stage2_base_report, indent=2))

free_memory("stage2_base_model", "stage2_base_tokenizer")

07:01:31 | INFO | QwenCodeGen | Generating Stage 2 base-model predictions ...
07:02:21 | INFO | QwenCodeGen | [base] evaluated 10/50
07:03:03 | INFO | QwenCodeGen | [base] evaluated 20/50
07:03:54 | INFO | QwenCodeGen | [base] evaluated 30/50
07:04:59 | INFO | QwenCodeGen | [base] evaluated 40/50
07:05:49 | INFO | QwenCodeGen | [base] evaluated 50/50
07:05:49 | INFO | QwenCodeGen | Stage 2 base-model evaluation finished on 50 samples (dataset=CodeXGLUE-Java-CS).
07:05:50 | INFO | QwenCodeGen | GPU [after free_memory] Tesla T4 | 2.74 GB allocated, 2.81 GB reserved, 15.6 GB total



===== Stage 2 (Java -> C#) Base Model Evaluation Report =====
Dataset: CodeXGLUE-Java-CS | Eval samples: 50
{
  "BLEU": 27.5104,
  "CodeBLEU": {
    "codebleu": 0.3627,
    "ngram_match_score": 0.0573,
    "weighted_ngram_match_score": 0.2126,
    "syntax_match_score": 0.5161,
    "dataflow_match_score": 0.6645
  },
  "ExactMatch": 0.0,
  "SyntaxAccuracy": 0.0,
  "AST_Similarity": 0.8403,
  "CompilationRate": {
    "value": 0.0,
    "method": "syntax-proxy(tree-sitter)"
  }
}


In [50]:
def _s2_get_bleu(rpt):
    return rpt.get("BLEU", float("nan"))


def _s2_get_codebleu(rpt):
    cb = rpt.get("CodeBLEU", {})
    return cb.get("codebleu", float("nan")) if isinstance(cb, dict) else float("nan")


def _s2_get_metric(rpt, key):
    v = rpt.get(key)
    return float("nan") if v is None else float(v)


def _s2_get_compilation(rpt):
    cr = rpt.get("CompilationRate", {})
    if isinstance(cr, dict):
        v = cr.get("value")
        return float("nan") if v is None else float(v)
    return float("nan")


def _s2_fmt(v):
    return f"{v:.4f}" if not (isinstance(v, float) and math.isnan(v)) else "N/A"


def _s2_diff(ft_v, base_v):
    if any(isinstance(x, float) and math.isnan(x) for x in (ft_v, base_v)):
        return "N/A"
    diff = ft_v - base_v
    return f"+{diff:.4f}" if diff >= 0 else f"{diff:.4f}"


stage2_comparison_rows = [
    ("BLEU", _s2_get_bleu(stage2_base_report), _s2_get_bleu(stage2_report)),
    ("CodeBLEU", _s2_get_codebleu(stage2_base_report),
     _s2_get_codebleu(stage2_report)),
    ("Exact Match", _s2_get_metric(stage2_base_report, "ExactMatch"),
     _s2_get_metric(stage2_report, "ExactMatch")),
    ("Syntax Accuracy", _s2_get_metric(stage2_base_report, "SyntaxAccuracy"),
     _s2_get_metric(stage2_report, "SyntaxAccuracy")),
    ("AST Similarity", _s2_get_metric(stage2_base_report, "AST_Similarity"),
     _s2_get_metric(stage2_report, "AST_Similarity")),
    ("Compilation Rate", _s2_get_compilation(stage2_base_report),
     _s2_get_compilation(stage2_report)),
]

_W = 26
print(f"\n{'=' * 74}")
print("  Java -> C#  |  Base vs Fine-tuned Comparison (Stage 2)")
print("=" * 74)
print(f"  {'Metric':{_W}}  {'Base Model':>12}  {'Fine-tuned':>12}  {'Improvement':>12}")
print(f"  {'-' * _W}  {'-' * 12}  {'-' * 12}  {'-' * 12}")
for _name, _bv, _fv in stage2_comparison_rows:
    print(
        f"  {_name:{_W}}  {_s2_fmt(_bv):>12}  {_s2_fmt(_fv):>12}  {_s2_diff(_fv, _bv):>12}")
print("=" * 74)

stage2_evaluation = {"fine_tuned": stage2_report, "base": stage2_base_report}


  Java -> C#  |  Base vs Fine-tuned Comparison (Stage 2)
  Metric                        Base Model    Fine-tuned   Improvement
  --------------------------  ------------  ------------  ------------
  BLEU                             27.5104       83.2815      +55.7711
  CodeBLEU                          0.3627        0.7376       +0.3749
  Exact Match                       0.0000        0.4000       +0.4000
  Syntax Accuracy                   0.0000        0.9200       +0.9200
  AST Similarity                    0.8403        0.9817       +0.1414
  Compilation Rate                  0.0000        0.9200       +0.9200


## 11. Model Export

In [ ]:
def get_hf_token() -> Optional[str]:
    """Return a HF token from the first available source, or None."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_WRITE_TOKEN")
    except Exception:
        pass
    token = os.environ.get("HF_WRITE_TOKEN") or os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_WRITE_TOKEN")
    except Exception:
        pass
    from getpass import getpass
    return getpass("Enter your Hugging Face token: ")

In [ ]:
def save_all_artifacts() -> str:
    """Save both adapters, tokenizers, config + metrics. Returns the run directory."""
    run_dir = os.path.join(CFG.SAVE_DIR, f"run_{TIMESTAMP}")
    stage1_dir = os.path.join(run_dir, "stage1_nl2java_adapter_V5")
    stage2_dir = os.path.join(run_dir, "stage2_java2csharp_adapter_V5")
    os.makedirs(run_dir, exist_ok=True)

    try:
        model.save_pretrained(stage1_dir)
        tokenizer.save_pretrained(stage1_dir)
        model_s2.save_pretrained(stage2_dir)
        tokenizer_s2.save_pretrained(stage2_dir)
        logger.info("Saved Stage 1 + Stage 2 adapters and tokenizers.")
    except Exception:
        logger.exception("Failed to save model adapters / tokenizers.")
        raise

    training_configuration = {
        "timestamp": TIMESTAMP,
        "base_model": CFG.MODEL_NAME,
        "stage1": {
            "task": "NL -> Java",
            "dataset": f"{CFG.DATASET_NAME}/{CFG.DATASET_CONFIG}",
            "lora": {"r": CFG.LORA_R, "lora_alpha": CFG.LORA_ALPHA,
                     "lora_dropout": CFG.LORA_DROPOUT, "bias": CFG.LORA_BIAS},
            "max_seq_length": CFG.MAX_LENGTH,
            "epochs": CFG.EPOCHS,
            "train_samples": CFG.STAGE1_TRAIN_SAMPLES,
            "hf_repo": CFG.HF_REPO,
            "evaluation": globals().get("stage1_evaluation"),
        },
        "stage2": {
            "task": "Java -> C#",
            "dataset": DATASET_NAME,
            "lora": {"r": CFG.LORA_R, "lora_alpha": CFG.STAGE2_LORA_ALPHA,
                     "lora_dropout": CFG.LORA_DROPOUT, "bias": CFG.LORA_BIAS},
            "max_seq_length": STAGE2_MAX_SEQ_LEN,
            "optimizer": stage2_config.optim,
            "fp16": stage2_config.fp16,
            "epochs": stage2_config.num_train_epochs,
            "per_device_train_batch_size": stage2_config.per_device_train_batch_size,
            "gradient_accumulation_steps": stage2_config.gradient_accumulation_steps,
            "learning_rate": stage2_config.learning_rate,
            "train_samples": len(train_pairs),
            "val_samples": len(val_pairs),
            "hf_repo": CFG.HF_REPO_STAGE2,
            "evaluation": globals().get("stage2_evaluation", stage2_report),
        },
    }

    config_path = os.path.join(run_dir, "training_configuration.json")
    with open(config_path, "w", encoding="utf-8") as fh:
        json.dump(training_configuration, fh, indent=2, default=str)

    logger.info("Saved configuration + metrics to %s", config_path)
    logger.info("Artifacts written to:\n - %s\n - %s\n - %s",
                stage1_dir, stage2_dir, config_path)
    return run_dir


SAVE_RUN_DIR = save_all_artifacts()

07:09:12 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:09:12 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
07:09:12 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:09:12 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
07:09:13 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:09:13 | INFO | httpx | HTTP Request: HEAD https://huggingface.c

In [ ]:
PUSH_BOTH_STAGES_TO_HUB = True
hf_token = get_hf_token()

if PUSH_BOTH_STAGES_TO_HUB and hf_token:
    try:
        logger.info("Pushing merged Stage 1 model to '%s' ...", CFG.HF_REPO)
        model.push_to_hub_merged(
            CFG.HF_REPO, tokenizer, save_method="merged_16bit", token=hf_token)
        logger.info("Stage 1 push complete: %s", CFG.HF_REPO)
    except Exception:
        logger.exception("Failed to push Stage 1 merged model to the Hub.")
        raise

    try:
        logger.info("Pushing merged Stage 2 model to '%s' ...",
                    CFG.HF_REPO_STAGE2)
        model_s2.push_to_hub_merged(
            CFG.HF_REPO_STAGE2, tokenizer_s2, save_method="merged_16bit", token=hf_token)
        logger.info("Stage 2 push complete: %s", CFG.HF_REPO_STAGE2)
    except Exception:
        logger.exception("Failed to push Stage 2 merged model to the Hub.")
        raise

    logger.info("Both stages merged and pushed:\n - %s\n - %s",
                CFG.HF_REPO, CFG.HF_REPO_STAGE2)
else:
    logger.warning("Skipping Hub push (PUSH_BOTH_STAGES_TO_HUB=%s, token_present=%s).",
                   PUSH_BOTH_STAGES_TO_HUB, bool(hf_token))

07:09:35 | INFO | QwenCodeGen | Pushing merged Stage 1 model to 'shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5' ...
07:09:35 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:09:35 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:09:35 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct/b9f2b422413d50e652bf90364af0efe50b48f92e/config.json "HTTP/1.1 200 OK"
07:09:35 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
07:09:35 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
07:09:

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
07:09:37 | WARNING | huggingface_hub.hf_api | No files have been modified since last commit. Skipping to prevent empty commit.
07:09:37 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/revision/main "HTTP/1.1 200 OK"


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
07:11:06 | WARNING | huggingface_hub.hf_api | No files have been modified since last commit. Skipping to prevent empty commit.
07:11:06 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/revision/main "HTTP/1.1 200 OK"
Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:35<00:00, 35.42s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5`


07:11:08 | INFO | QwenCodeGen | Stage 2 push complete: shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5
07:11:08 | INFO | QwenCodeGen | Both stages merged and pushed:
 - shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5
 - shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5
